# Quantum-Assisted Lightweight Deep Learning for Sensor-Aware 3D Hybrid Beamforming in RIS-Aided THz MIMO

**Base paper:** X. Wang, Z. Lin, F. Lin, L. Hanzo, *Hybrid 3D Beamforming Relying on Sensor-Based Training and Channel Estimation for RIS-Aided THz MIMO Systems* (arXiv 2201.10757).

This notebook re-implements the base system (array-of-subarray BS, sub-RIS, UWB sensing, PBA, MMSE digital precoder) and applies the four requested replacements:

| # | Base paper | Replaced by | Notebook section |
|---|-----------|-------------|------------------|
| 1 | Sec. V - sensor-based channel estimation | **Lightweight geometry-aware channel estimation** (UWB 3D positioning *retained*, geometric features *retained + extended*, depthwise-separable CNN, GRU temporal prediction, complex channel via real/imag heads) | 3-4 |
| 2 | Sec. V-C - Precise Beamforming Algorithm (PBA, Algorithm 1) | **UL-JBA**: shortlist of Rx beams + RIS configurations, local alternatives, stop when rate gain < threshold | 5 |
| 3 | Eqs. (39)-(40) as the only phase selector | **Q-RPO**: closed-form phases kept as the start; reduced discrete phase choices -> binary problem -> **QAOA / VQC**, compared with classical optimisers on the *same* candidate space | 6-7 |
| 4 | Sec. V-D - complexity analysis | **Computational + quantum-execution + inference cost** for PBA / lightweight-only / quantum-assisted / full framework | 9 |

Everything runs with `numpy / scipy / matplotlib / torch`. The quantum circuits are simulated exactly with a NumPy state-vector simulator (no quantum SDK needed). Set `FAST = False` in the first cell for paper-scale statistics.

### Modelling choices you should know about (read once)
* **Scale.** Arrays are reduced for a CPU run: BS subarray 8x8, user 8x8, sub-RIS 12x12 (144 passive elements), K = 4 users, 350 GHz, N0 = -75 dBm, 1 GHz (paper Table III values where given). Distances are 1-2 m so that rates are in a meaningful range with these small RIS apertures.
* **Angle convention.** The paper writes the RIS-user angle as `arcsin(z/d)` inside `sin(.)` terms; I use the standard UPA convention (polar angle from the array normal, direction cosines `x/d, y/d`), which is the physically consistent form of the same steering vector.
* **Cascaded channel.** `H = sum_l c_l h_l a_l^T` (LOS + two NLOS rays at -12/-22 dB, matching the paper's >10/>20 dB statement), `G w_k = c_1 a_sa` with the BS beam aligned to the sub-RIS (Eq. 33). Per-hop LOS gain follows Eq. (10).
* **MMSE precoder.** Implemented in the standard regularised form `F = T^H (T T^H + K sigma^2/P I)^-1` (the paper's Eq. 42 as typeset is garbled). Cross-user leakage terms `T[k,i]` are computed from the actual channels, not assumed zero.
* **UWB anchors** sit at (+-0.5, +-0.5, 0) m around the RIS centre so the multilateration geometry is non-degenerate for a small RIS; ranging error is Gaussian (3 cm).
* **Timing.** Beams are applied one UWB report *after* the last measurement (`horizon = 1`), so the estimator has to *predict* (this is where the GRU matters) and the original closed-form starts from a stale position.
* **Probing.** A probe is a pilot-based complex measurement of the effective gain with realistic noise. Q-RPO obtains its cost function from `tiles + 1` such probes (tile-wise pi-flip training), not from the true channel.

## 1. Configuration

In [ ]:
import os, time, math, warnings, itertools
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass
from scipy.optimize import minimize
warnings.filterwarnings('ignore')
np.set_printoptions(precision=4, suppress=True)

SEED = 7
FAST = True          # True: small data / few test snapshots (minutes on CPU). False: paper-scale statistics.
rng = np.random.default_rng(SEED)

@dataclass
class Cfg:
    # --- THz / array parameters (Table III of the base paper, scaled to a CPU-friendly size) ---
    fc: float = 350e9            # centre frequency [Hz]
    mu: float = 1e-3             # molecular absorption coefficient mu(f) [1/m] (350 GHz window)
    K: int = 4                   # users = sub-RISs = BS subarrays = RF chains
    mt: int = 8; nt: int = 8     # BS subarray UPA
    mr: int = 8; nr: int = 8     # user UPA
    ms: int = 12; ns: int = 12   # sub-RIS UPA (144 passive elements)
    noise_dbm: float = -75.0     # noise power N0 (paper)
    bits: int = 2                # RIS phase resolution (b-bit PIN-diode / varactor control)
    bs0: tuple = (-1.0, -0.8, 1.1)   # BS centre [m]; RIS centre is the origin, RIS normal = +z
    # --- UWB sensing ---
    uwb_sigma: float = 0.03      # ranging std [m]
    anchor_half: float = 0.5     # four UWB anchors at (+-a, +-a, 0)
    r_e: float = 0.15            # PBA assumed positioning-error sphere radius [m]
    # --- mobility / sequence ---
    T: int = 6                   # history length for the GRU
    dt: float = 0.1              # UWB report period [s]
    horizon: int = 1             # predict the channel one report-period ahead
    # --- NLOS rays in the *true* channel (paper: -10 dB and -20 dB w.r.t. LOS) ---
    n_nlos: int = 2
    nlos_db: tuple = (-12.0, -22.0)
    # --- Q-RPO tiling: tile_r x tile_c tiles, 2 bits (4 phase offsets) per tile -> n_qubits = 2*tiles ---
    tile_r: int = 2; tile_c: int = 3
    # --- probing ---
    probe_dbm: float = 40.0      # pilot transmit power used during beam probing
    probe_len: int = 32          # pilot symbols averaged per probe
cfg = Cfg()

C0 = 3e8
LAM = C0 / cfg.fc
K0 = 2 * np.pi / LAM
Ms, Nr, Nt = cfg.ms * cfg.ns, cfg.mr * cfg.nr, cfg.mt * cfg.nt
SIG2 = 10 ** (cfg.noise_dbm / 10) * 1e-3          # noise power [W]
NT = cfg.tile_r * cfg.tile_c                      # number of RIS tiles
NQ = 2 * NT                                       # qubits for Q-RPO
print(f"lambda = {LAM*1e3:.3f} mm | RIS elements/sub-RIS = {Ms} | user antennas = {Nr} | BS subarray = {Nt} | qubits = {NQ}")

## 2. System model: steering vectors, cascaded channel, path loss, codebooks
Implements Eqs. (1)-(23): UPA steering (Eqs. 11-14), THz LOS gain (Eq. 10), near/far-field path loss (Eqs. 22-23), orthogonality spacing (Eqs. 16-17), far-field boundary (Eq. 20), RIS phase closed form (Eq. 39). The receive/RIS codebook primitives used by PBA and UL-JBA are defined here too. A sanity check confirms the simulated coherent gain equals `|beta1 beta2| sqrt(Nt Nr) Ms`.

In [ ]:
def grid_pos(m, n, sp):
    x = (np.arange(m) - (m - 1) / 2) * sp
    y = (np.arange(n) - (n - 1) / 2) * sp
    X, Y = np.meshgrid(x, y, indexing='ij')
    return np.stack([X.ravel(), Y.ravel()], 1)

RPOS = grid_pos(cfg.ms, cfg.ns, LAM / 2)     # sub-RIS element coordinates (spacing lambda/2)
UPOS = grid_pos(cfg.mr, cfg.nr, LAM / 2)     # user UPA
BPOS = grid_pos(cfg.mt, cfg.nt, LAM / 2)     # BS subarray UPA

def unit(x):
    return x / np.linalg.norm(x, axis=-1, keepdims=True)

def steer(u, pos):
    # UPA steering vector for a unit direction u (only the x,y direction cosines matter for a planar array)
    ph = K0 * (u[..., 0:1] * pos[:, 0] + u[..., 1:2] * pos[:, 1])
    return np.exp(1j * ph) / np.sqrt(pos.shape[0])

def beta(d):
    # LOS complex gain, Eq. (10): |beta|^2 = c^2 / (4 pi d f)^2 * exp(-mu d)
    return C0 / (4 * np.pi * d * cfg.fc) * np.exp(-cfg.mu * d / 2) * np.exp(-1j * K0 * d)

def tangent(u):
    ref = np.array([0.0, 0.0, 1.0])
    e1 = unit(np.cross(u, ref)); e2 = np.cross(u, e1)
    return e1, e2

# ---- paper Eq. (22)/(23): near-/far-field cascaded path loss, Eq. (16)-(17): optimal sub-array spacing, Eq. (20): boundary
def pl_near(d1, d2):
    return C0**2 / ((4 * np.pi * cfg.fc) ** 2 * (d1 + d2) ** 2) * np.exp(-cfg.mu * (d1 + d2))
def pl_far(d1, d2):
    return C0**2 / ((4 * np.pi * cfg.fc) ** 2 * d1**2 * d2**2) * np.exp(-cfg.mu * (d1 + d2))
def spacing_opt(d1, N, q=1):
    return np.sqrt(q * d1 * LAM / N)
def far_field_boundary(Nsub, m, n):
    return Nsub**2 * m * n * LAM / 2

# ---- BS -> RIS leg (static, known geometry): vector s_k = G_k w_k with w_k = a_t (Eq. 33), includes the BS array gain
BS_ctr = np.array(cfg.bs0)
def bs_vec(k):
    pos = BS_ctr + np.array([0.0, 0.12 * (k - (cfg.K - 1) / 2), 0.0])
    d1 = np.linalg.norm(pos)
    return beta(d1) * np.sqrt(Nt * Ms) * steer(pos / d1, RPOS)
S_BS = np.stack([bs_vec(k) for k in range(cfg.K)])          # (K, Ms)
D1 = np.linalg.norm(BS_ctr)

# ---- tiles for Q-RPO
def tile_index():
    r = np.arange(cfg.ms)[:, None] // (cfg.ms // cfg.tile_r)
    c = np.arange(cfg.ns)[None, :] // (cfg.ns // cfg.tile_c)
    return (r * cfg.tile_c + c).ravel()
TILE = tile_index()

# ---- RIS-user channel with LOS + NLOS rays:  H = sum_l c_l h_l a_l^T  (h_l: user side, a_l: RIS side)
def build_channel(P, k_idx, rg, nlos=True):
    N = len(P); d = np.linalg.norm(P, axis=1); u0 = P / d[:, None]
    L = 1 + (cfg.n_nlos if nlos else 0)
    dirs = np.repeat(u0[:, None, :], L, 1)
    c = np.zeros((N, L), complex)
    c[:, 0] = beta(d) * np.sqrt(Nr * Ms)
    e1, e2 = tangent(u0)
    for l in range(1, L):
        ang = rg.uniform(0.25, 0.6, N) * rg.choice([-1, 1], N); az = rg.uniform(0, 2 * np.pi, N)
        off = ang[:, None] * (np.cos(az)[:, None] * e1 + np.sin(az)[:, None] * e2)
        dirs[:, l] = unit(u0 + off)
        c[:, l] = c[:, 0] * 10 ** (cfg.nlos_db[l - 1] / 20) * np.exp(1j * rg.uniform(0, 2 * np.pi, N))
    return dict(P=P, dirs=dirs, c=c, h=np.conj(steer(dirs, UPOS)), a=steer(dirs, RPOS), s=S_BS[np.asarray(k_idx)])

def pick(ch, i):
    return {k: v[i] for k, v in ch.items()}

def gain(ch, v, q, s=None):
    # g = v^H H O G w  = sum_l c_l (v^H h_l) (a_l o s)^T q     (single user dict from pick())
    s = ch['s'] if s is None else s
    return np.sum(ch['c'] * (ch['h'] @ np.conj(v)) * ((ch['a'] * s) @ q))

# ---- codebook primitives
def rx_beam(u):                      # receive beam matched to a LOS direction u
    return np.conj(steer(u, UPOS))
def ris_phase(u, s):                 # closed-form RIS phases (Eq. 39/40 structure) for LOS direction u
    return np.exp(-1j * (np.angle(steer(u, RPOS)) + np.angle(s)))
def ris_phase_from_u(u_est, s):      # Eq. (39): q = [(H^T v*) o a_sa]^*  (unit modulus)
    return np.exp(-1j * (np.angle(u_est) + np.angle(s)))
def quant(q, bits=None):
    bits = cfg.bits if bits is None else bits
    st = 2 * np.pi / 2 ** bits
    return np.exp(1j * np.round(np.angle(q) / st) * st)

def snr_to_rate(g, p_tot_dbm):       # single-user (interference-free) rate, power P/K per user
    P = 10 ** (p_tot_dbm / 10) * 1e-3
    return np.log2(1 + P / cfg.K * np.abs(g) ** 2 / SIG2)

d1_, d2_ = D1, 1.5
print(f"d1={d1_:.2f} m | far-field boundary of one sub-RIS D={far_field_boundary(1,cfg.ms,cfg.ns)*1e3:.1f} mm (Eq. 20 with N=1)")
print(f"Eq.(22) near-field PL = {10*np.log10(pl_near(d1_,d2_)):.1f} dB | Eq.(23) far-field PL = {10*np.log10(pl_far(d1_,d2_)):.1f} dB")
print(f"Theorem-1 optimal sub-RIS spacing (q=1, N=2) = {spacing_opt(d1_,2)*1e3:.2f} mm")
# quick sanity: continuous genie gain on a LOS-only channel
_p = np.array([[0.6, 0.7, 1.0]]); _ch = pick(build_channel(_p, [0], np.random.default_rng(0), nlos=False), 0)
_u = _p[0] / np.linalg.norm(_p[0]); _g = gain(_ch, rx_beam(_u), ris_phase(_u, _ch['s']))
print(f"sanity: |g|={abs(_g):.3e}  expected |beta1 beta2| sqrt(Nt Nr) Ms = {abs(beta(D1)*beta(np.linalg.norm(_p)))*np.sqrt(Nt*Nr)*Ms:.3e}")

## 3. Retained blocks of Section V: UWB 3D positioning and geometric feature extraction
* **UWB multilateration** (Eq. 24) by batched Gauss-Newton on four ranges with error `eps`.
* **Geometric channel features** (Eqs. 25-32, *retained*): azimuth/polar angles, distance, cascaded LOS path loss.
* **Extended features** (*new*): range residuals (a free quality indicator of the position fix), sin/cos angle encodings, and a **5-channel RIS-aperture geometry map** (Re/Im of the geometry-predicted RIS steering response, x/y coordinates, distance) that feeds the CNN.
* Users move (random-acceleration model); a window of `T = 6` UWB reports is the model input and the **channel one report ahead** is the target.

In [ ]:
a_ = cfg.anchor_half
ANCHORS = np.array([[a_, a_, 0.0], [a_, -a_, 0.0], [-a_, a_, 0.0], [-a_, -a_, 0.0]])

def uwb_multilaterate(P, rg, sigma=None):
    # Eq. (24): d~_i = ||P_U - P_si|| + eps ; solved by batched Gauss-Newton least squares
    sigma = cfg.uwb_sigma if sigma is None else sigma
    sh = P.shape[:-1]; P = P.reshape(-1, 3)
    dm = np.linalg.norm(P[:, None, :] - ANCHORS[None], axis=2) + rg.normal(0, sigma, (len(P), 4))
    p = np.zeros_like(P); p[:, 2] = dm.mean(1)                    # initial guess above the RIS plane
    for _ in range(12):
        diff = p[:, None, :] - ANCHORS[None]; r = np.linalg.norm(diff, axis=2)
        J = diff / r[..., None]; res = r - dm
        A = np.einsum('nik,nil->nkl', J, J) + 1e-9 * np.eye(3)
        b = np.einsum('nik,ni->nk', J, res)
        p = p - np.linalg.solve(A, b[..., None])[..., 0]
        p[:, 2] = np.abs(p[:, 2])
    res = dm - np.linalg.norm(p[:, None, :] - ANCHORS[None], axis=2)   # range residuals (a free quality indicator)
    return p.reshape(*sh, 3), res.reshape(*sh, 4)

def make_traj(N, rg):
    r = rg.uniform(0.8, 2.0, N); pol = rg.uniform(0.35, 1.1, N); az = rg.uniform(0, 2 * np.pi, N)
    p0 = np.stack([r * np.sin(pol) * np.cos(az), r * np.sin(pol) * np.sin(az), r * np.cos(pol)], 1)
    sp = rg.uniform(0.1, 0.6, N); hd = rg.uniform(0, 2 * np.pi, N)
    vel = np.stack([sp * np.cos(hd), sp * np.sin(hd), rg.normal(0, 0.05, N)], 1)
    P = np.zeros((N, cfg.T + cfg.horizon, 3)); P[:, 0] = p0
    for t in range(1, cfg.T + cfg.horizon):
        vel = vel + rg.normal(0, 0.08, (N, 3))
        P[:, t] = P[:, t - 1] + vel * cfg.dt
    P[..., 2] = np.maximum(P[..., 2], 0.3)
    return P

# ---- geometric channel-feature extraction (Eqs. 25-32 retained, then extended)
def geo_features(phat, res):
    d = np.linalg.norm(phat, axis=-1, keepdims=True); u = phat / d
    az = np.arctan2(u[..., 1], u[..., 0]); pol = np.arccos(np.clip(u[..., 2], -1, 1))
    pl_db = 20 * np.log10(np.abs(beta(d[..., 0])) * np.abs(beta(D1)) + 1e-30)          # cascaded LOS path loss (Eq. 10)
    f = [phat / 2.0, d / 2.0, np.sin(az)[..., None], np.cos(az)[..., None], np.sin(pol)[..., None], np.cos(pol)[..., None],
         res / cfg.uwb_sigma / 2.0, ((pl_db + 190.0) / 20.0)[..., None]]
    return np.concatenate(f, -1).astype(np.float32)

N_GEO = 3 + 1 + 2 + 2 + 4 + 1
_gx, _gy = np.meshgrid(np.linspace(-1, 1, cfg.ms), np.linspace(-1, 1, cfg.ns), indexing='ij')

def spatial_maps(phat):
    # per-step 5-channel RIS-aperture map: Re/Im of the geometry-predicted RIS steering response, x/y coordinates, distance
    u = unit(phat); a = steer(u, RPOS) * np.sqrt(Ms)                          # unit-modulus entries
    lead = phat.shape[:-1]
    re = a.real.reshape(*lead, cfg.ms, cfg.ns); im = a.imag.reshape(*lead, cfg.ms, cfg.ns)
    d = (np.linalg.norm(phat, axis=-1) / 2.0)[..., None, None] * np.ones((cfg.ms, cfg.ns))
    gx = np.broadcast_to(_gx, re.shape); gy = np.broadcast_to(_gy, re.shape)
    return np.stack([re, im, gx, gy, d], axis=-3).astype(np.float32)           # (..., 5, ms, ns)

def make_dataset(N, seed):
    rg = np.random.default_rng(seed)
    P = make_traj(N, rg)
    phat, res = uwb_multilaterate(P[:, :cfg.T], rg)
    k_idx = np.arange(N) % cfg.K
    ch = build_channel(P[:, cfg.T + cfg.horizon - 1], k_idx, rg)               # true channel at the application instant
    # learning target: effective RIS-side channel seen through the matched receive beam, unit-modulus-scaled
    afr = np.einsum('nm,nlm->nl', np.conj(ch['h'][:, 0]), ch['h'])
    u = np.einsum('nl,nl,nlm->nm', ch['c'], afr, ch['a'])
    u = u / np.linalg.norm(u, axis=1, keepdims=True) * np.sqrt(Ms)
    return dict(P=P, phat=phat, res=res, k_idx=k_idx, ch=ch, u=u,
                geo=geo_features(phat, res), sp=spatial_maps(phat))

t0 = time.time()
tr = make_dataset(2400 if FAST else 8000, SEED)
va = make_dataset(300, SEED + 1)
te = make_dataset(400 if FAST else 1200, SEED + 2)       # K users per snapshot -> 100 / 300 snapshots
print(f"datasets built in {time.time()-t0:.1f}s | train {tr['sp'].shape} | test users {len(te['P'])}")

# ---- UWB positioning-error statistics (last measurement vs *current* true position)
err_meas = np.linalg.norm(te['phat'] - te['P'][:, :cfg.T], axis=-1)
err_stale = np.linalg.norm(te['phat'][:, -1] - te['P'][:, -1], axis=-1)
err_apply = np.linalg.norm(te['phat'][:, -1] - te['P'][:, cfg.T + cfg.horizon - 1], axis=-1)
print(f"UWB error [cm]: mean {100*err_meas.mean():.1f}, 90th pct {100*np.percentile(err_meas,90):.1f}, max {100*err_meas.max():.1f}")
print(f"error at the instant the beams are applied (stale by {cfg.horizon} report): mean {100*err_apply.mean():.1f} cm, 90th pct {100*np.percentile(err_apply,90):.1f} cm")
print(f"fraction of cases exceeding the PBA assumed sphere r_e={100*cfg.r_e:.0f} cm: {np.mean(err_apply>cfg.r_e):.2%}")

## 4. Change 1 - Section V: Lightweight geometry-aware channel estimation using sensor-assisted deep learning
**Model (`LightEstimator`)**
1. *Spatial branch:* stem conv + two **depthwise-separable** blocks (3x3 depthwise + 1x1 pointwise) on the geometry map, global pooling -> compact spatial feature.
2. *Geometry branch:* MLP on the UWB/angle/path-loss/residual features.
3. *Temporal branch:* **GRU** over the last `T` reports -> predicts the channel at the next report instant.
4. *Complex reconstruction:* separate **real head and imaginary head** (`u = prior + Re/Im corrections`), so the output is a complex RIS-side channel vector (the quantity needed by Eq. 39).
5. *Auxiliary head:* position correction + log-variance (heteroscedastic), combined with **MC-dropout** to give the per-axis position uncertainty `sigma` that UL-JBA uses to size its search.

Loss = complex MSE + (1 - |cos|^2) + 0.1 x Gaussian NLL of the position correction.

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
torch.manual_seed(SEED); torch.set_num_threads(max(1, os.cpu_count() or 1))
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'

class DSConv(nn.Module):                       # depthwise 3x3 + pointwise 1x1 (MobileNet-style)
    def __init__(s, cin, cout):
        super().__init__()
        s.dw = nn.Conv2d(cin, cin, 3, padding=1, groups=cin, bias=False)
        s.pw = nn.Conv2d(cin, cout, 1, bias=False)
        s.bn = nn.BatchNorm2d(cout); s.act = nn.ReLU6()
    def forward(s, x): return s.act(s.bn(s.pw(s.dw(x))))

class StdConv(nn.Module):                      # dense 3x3 conv (used only for the ablation / complexity comparison)
    def __init__(s, cin, cout):
        super().__init__()
        s.cv = nn.Conv2d(cin, cout, 3, padding=1, bias=False); s.bn = nn.BatchNorm2d(cout); s.act = nn.ReLU6()
    def forward(s, x): return s.act(s.bn(s.cv(x)))

class LightEstimator(nn.Module):
    # Section V (new): geometry-aware sensor-assisted estimator
    #   spatial branch : depthwise-separable CNN on the RIS-aperture geometry map
    #   geometry branch: MLP on UWB position / angle / path-loss / range-residual features
    #   temporal branch: GRU over the last T reports  -> channel at the next report instant
    #   heads          : real head + imaginary head (complex channel), position correction + log-variance
    def __init__(s, ms, ns, n_geo, use_gru=True, ds=True, ch=(16, 32, 32), hid=64, p_drop=0.1):
        super().__init__()
        Blk = DSConv if ds else StdConv
        s.stem = nn.Sequential(nn.Conv2d(5, ch[0], 3, padding=1, bias=False), nn.BatchNorm2d(ch[0]), nn.ReLU6())
        s.b1, s.b2 = Blk(ch[0], ch[1]), Blk(ch[1], ch[2])
        s.fc_sp = nn.Sequential(nn.Linear(ch[2], 32), nn.ReLU())
        s.geo = nn.Sequential(nn.Linear(n_geo, 32), nn.ReLU())
        s.use_gru = use_gru
        if use_gru: s.gru = nn.GRU(64, hid, batch_first=True)
        else: s.mlp = nn.Sequential(nn.Linear(64, hid), nn.ReLU())
        s.drop = nn.Dropout(p_drop)
        s.re = nn.Linear(hid, ms * ns); s.im = nn.Linear(hid, ms * ns); s.pos = nn.Linear(hid, 6)
        for l in (s.re, s.im): nn.init.zeros_(l.weight); nn.init.zeros_(l.bias)      # start from the geometric prior
    def forward(s, sp, geo):
        B, T = sp.shape[:2]
        x = s.b2(s.b1(s.stem(sp.flatten(0, 1))))
        f = s.fc_sp(x.mean((2, 3))).view(B, T, -1)
        z = torch.cat([f, s.geo(geo)], -1)
        if s.use_gru: h = s.gru(z)[0][:, -1]
        else: h = s.mlp(z[:, -1])
        h = s.drop(h)
        prior = sp[:, -1, :2].flatten(2)                                   # (B,2,Ms) Re/Im of the last geometric prior
        re = prior[:, 0] + s.re(h); im = prior[:, 1] + s.im(h)             # complex output = prior + learned correction
        po = s.pos(h)
        return re, im, po[:, :3], po[:, 3:]                                # Delta-p (x0.1 m), log-sigma (x0.1 m)

def to_t(d):
    return (torch.tensor(d['sp']), torch.tensor(d['geo']),
            torch.tensor(np.stack([d['u'].real, d['u'].imag], 1), dtype=torch.float32),
            torch.tensor((d['P'][:, cfg.T + cfg.horizon - 1] - d['phat'][:, -1]) / 0.1, dtype=torch.float32))
TR, VA = to_t(tr), to_t(va)

def loss_fn(out, u, dp):
    re, im, mp, ls = out
    mse = ((re - u[:, 0]) ** 2 + (im - u[:, 1]) ** 2).mean()
    ip_r = (re * u[:, 0] + im * u[:, 1]).sum(1); ip_i = (im * u[:, 0] - re * u[:, 1]).sum(1)
    cs = 1 - (ip_r ** 2 + ip_i ** 2) / (((re ** 2 + im ** 2).sum(1)) * ((u ** 2).sum((1, 2))) + 1e-9)   # 1-|<u_hat,u>|^2/(|u_hat|^2|u|^2)
    nll = (0.5 * ((mp - dp) ** 2 * torch.exp(-2 * ls)) + ls).sum(1).mean()
    return mse + cs.mean() + 0.1 * nll, mse.item(), cs.mean().item()

def train_estimator(use_gru=True, ds=True, epochs=14, bs=128, lr=2e-3, verbose=True):
    m = LightEstimator(cfg.ms, cfg.ns, N_GEO, use_gru, ds).to(DEV)
    opt = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=1e-4)
    n = TR[0].shape[0]; steps = epochs * math.ceil(n / bs)
    sch = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=steps)
    hist = []
    for ep in range(epochs):
        m.train(); perm = torch.randperm(n); tl = 0
        for i in range(0, n, bs):
            idx = perm[i:i + bs]
            sp, geo, u, dp = [t[idx].to(DEV) for t in TR]
            l, _, _ = loss_fn(m(sp, geo), u, dp)
            opt.zero_grad(); l.backward(); nn.utils.clip_grad_norm_(m.parameters(), 2.0); opt.step(); sch.step(); tl += l.item() * len(idx)
        m.eval()
        with torch.no_grad():
            sp, geo, u, dp = [t.to(DEV) for t in VA]
            vl, vm, vc = loss_fn(m(sp, geo), u, dp)
        hist.append((tl / n, vl.item() if torch.is_tensor(vl) else vl, vc))
        if verbose and (ep % 3 == 0 or ep == epochs - 1):
            print(f"  ep {ep+1:2d}/{epochs}  train {tl/n:.4f}  val {hist[-1][1]:.4f}  val (1-cos^2) {vc:.4f}")
    return m, hist

def n_params(m): return sum(p.numel() for p in m.parameters())

def make_predictor(m, mc=8):
    # MC-dropout inference: mean complex channel + position correction + total position uncertainty
    def predict(d):
        sp, geo = torch.tensor(d['sp']).to(DEV), torch.tensor(d['geo']).to(DEV)
        m.eval()
        for mod in m.modules():
            if isinstance(mod, nn.Dropout): mod.train()
        res, ims, mps, lss = [], [], [], []
        with torch.no_grad():
            for _ in range(mc):
                re, im, mp, ls = m(sp, geo)
                res.append(re.cpu().numpy()); ims.append(im.cpu().numpy()); mps.append(mp.cpu().numpy()); lss.append(ls.cpu().numpy())
        res, ims, mps, lss = map(np.array, (res, ims, mps, lss))
        u_hat = res.mean(0) + 1j * ims.mean(0)
        dpos = mps.mean(0) * 0.1
        sig = np.sqrt((np.exp(2 * lss).mean(0) + mps.var(0)) ) * 0.1             # aleatoric + epistemic, metres per axis
        return dict(u_hat=u_hat, p_est=d['phat'][:, -1] + dpos, sigma=sig)
    return predict

print('training the proposed lightweight estimator (DS-CNN + GRU + complex heads) ...')
t0 = time.time(); est_full, hist_full = train_estimator(True, True); print(f"  done in {time.time()-t0:.0f}s, parameters = {n_params(est_full):,}")
predict_full = make_predictor(est_full)

In [ ]:
def est_metrics(u_hat, d):
    u = d['u']; a = u_hat / (np.linalg.norm(u_hat, axis=1, keepdims=True) + 1e-12) * np.sqrt(Ms)
    nmse = np.mean(np.abs(a - u) ** 2, 1) / np.mean(np.abs(u) ** 2, 1)
    corr = np.abs(np.sum(np.conj(a) * u, 1)) / (np.linalg.norm(a, axis=1) * np.linalg.norm(u, axis=1))
    return nmse, corr

def prior_u(d):                       # original-paper estimator: closed-form from the (stale) UWB position, Eq. (40) structure
    return steer(unit(d['phat'][:, -1]), RPOS) * np.sqrt(Ms)

pred_te = predict_full(te)
rows = {}
nm, co = est_metrics(prior_u(te), te); rows['UWB geometry (original Sec. V)'] = (nm.mean(), co.mean(), np.linalg.norm(te['phat'][:, -1] - te['P'][:, cfg.T + cfg.horizon - 1], axis=1).mean())
nm, co = est_metrics(pred_te['u_hat'], te)
rows['Proposed DS-CNN + GRU'] = (nm.mean(), co.mean(), np.linalg.norm(pred_te['p_est'] - te['P'][:, cfg.T + cfg.horizon - 1], axis=1).mean())
print(f"{'estimator':34s} {'NMSE':>8s} {'|corr|':>8s} {'pos.err [cm]':>13s}")
for k, (a, b, c) in rows.items(): print(f"{k:34s} {a:8.4f} {b:8.4f} {100*c:13.2f}")
sig_hat = np.linalg.norm(pred_te['sigma'], axis=1); err = np.linalg.norm(pred_te['p_est'] - te['P'][:, cfg.T + cfg.horizon - 1], axis=1)
print(f"uncertainty calibration: corr(sigma_hat, |error|) = {np.corrcoef(sig_hat, err)[0,1]:.2f}")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
h = np.array(hist_full); ax[0].plot(h[:, 0], label='train'); ax[0].plot(h[:, 1], label='val'); ax[0].set_xlabel('epoch'); ax[0].set_ylabel('loss'); ax[0].legend(); ax[0].grid(alpha=.3); ax[0].set_title('Estimator training')
nm_p, co_p = est_metrics(prior_u(te), te); nm_n, co_n = est_metrics(pred_te['u_hat'], te)
ax[1].hist(co_p, 40, alpha=.6, label='UWB geometry (original)'); ax[1].hist(co_n, 40, alpha=.6, label='DS-CNN + GRU'); ax[1].set_xlabel('|correlation| with true RIS-side channel'); ax[1].legend(); ax[1].set_title('Channel reconstruction quality (test)')
plt.tight_layout(); plt.show()

**Ablation / compactness.** Standard convolution vs depthwise-separable, and GRU vs single-report input (parameters, multiply-accumulates, reconstruction quality).

In [ ]:
# ---- ablation + compactness: standard conv vs depthwise-separable, GRU vs single-step
def macs_estimator(ds=True, use_gru=True, ch=(16, 32, 32), hid=64):
    HW = cfg.ms * cfg.ns; T = cfg.T
    stem = HW * 5 * ch[0] * 9
    blk = (lambda ci, co: HW * (ci * 9 + ci * co)) if ds else (lambda ci, co: HW * ci * co * 9)
    cnn = (stem + blk(ch[0], ch[1]) + blk(ch[1], ch[2])) * T
    fcs = (ch[2] * 32 + N_GEO * 32) * T
    seq = 3 * (64 * hid + hid * hid) * T if use_gru else 64 * hid
    heads = hid * (2 * Ms + 6)
    return cnn + fcs + seq + heads

abl = {}
abl['Proposed (DS-CNN + GRU)'] = dict(model=est_full, pred=predict_full, ds=True, gru=True)
for name, (g_, d_) in {'Std-CNN + GRU': (True, False), 'DS-CNN, no GRU (last report only)': (False, True)}.items():
    print('training ablation:', name)
    mm, _ = train_estimator(g_, d_, epochs=10, verbose=False)
    abl[name] = dict(model=mm, pred=make_predictor(mm), ds=d_, gru=g_)
print(f"\n{'variant':36s} {'params':>9s} {'MACs/inf':>10s} {'NMSE':>8s} {'|corr|':>8s}")
for k, v in abl.items():
    nm, co = est_metrics(v['pred'](te)['u_hat'], te)
    v['nmse'], v['corr'] = nm.mean(), co.mean(); v['params'] = n_params(v['model']); v['macs'] = macs_estimator(v['ds'], v['gru'])
    print(f"{k:36s} {v['params']:9,d} {v['macs']:10,d} {v['nmse']:8.4f} {v['corr']:8.4f}")

## 5. Change 2 - Section V-C: Uncertainty-aware Lightweight Joint Beam Alignment (UL-JBA)
First the **original PBA** (Algorithm 1: sub-codebooks inside a fixed positioning-error sphere, alternate exhaustive Rx / RIS search until the measured gain stops improving) is implemented as the baseline. Then the modified **Algorithm 1 (UL-JBA)**:

1. Start from the closed-form configuration (Eq. 39 with the *estimated* complex channel; Eq. 34 receive beam).
2. Build a codebook region sized by the estimator's own uncertainty `sigma` (not a fixed `r_e`), and **predict a shortlist** of `S_v` receive beams and `S_w` RIS configurations ranked by expected array gain under that uncertainty.
3. Probe the shortlist (Rx beams with the initial RIS state, RIS candidates with the best Rx beam).
4. Evaluate **local alternatives** (+-1 codebook step in both tangent directions) around the incumbent.
5. **Stop** when the achievable-rate improvement of a round is below `eps` (0.05 bps/Hz).

Every probe is counted, since probes are the real cost of beam training.

In [ ]:
P_PROBE = 10 ** (cfg.probe_dbm / 10) * 1e-3
STEP_RX, STEP_RIS = 0.12, 0.08            # codebook resolution in direction-cosine units (~ half a beamwidth)

def new_ctr(): return dict(probes=0, iters=0, cls_flops=0.0)

def probe(ch, v, q, rg, ctr):
    # one pilot-based measurement of the effective gain g = v^H H O G w (complex, noisy, averaged over probe_len pilots)
    g = gain(ch, v, q); sd = np.sqrt(SIG2 / (cfg.probe_len * P_PROBE / cfg.K) / 2)
    ctr['probes'] += 1; ctr['cls_flops'] += 8 * (Nr + Ms) * (1 + cfg.n_nlos)
    return g + sd * (rg.normal() + 1j * rg.normal())

def rate_probe(gm): return np.log2(1 + P_PROBE / cfg.K * np.abs(gm) ** 2 / SIG2)

def dir_grid(u, radius, step):
    e1, e2 = tangent(u); n = max(1, int(np.ceil(radius / step))); out = []
    for i in range(-n, n + 1):
        for j in range(-n, n + 1):
            if i * i + j * j <= n * n: out.append(unit(u + step * (i * e1 + j * e2)))
    return np.array(out)

# ------------------------------------------------------------------ original Algorithm 1 (PBA) -- baseline
def pba(ch, p_hat, rg, ctr, r_e=None):
    r_e = cfg.r_e if r_e is None else r_e
    u0 = unit(p_hat); rad = r_e / np.linalg.norm(p_hat); s = ch['s']
    V = [rx_beam(x) for x in dir_grid(u0, rad, STEP_RX)]                     # sub-codebook inside the UWB error sphere
    W = [quant(ris_phase(x, s)) for x in dir_grid(u0, rad, STEP_RIS)]
    v, q = rx_beam(u0), quant(ris_phase(u0, s)); best = np.abs(probe(ch, v, q, rg, ctr))
    for _ in range(8):                                                         # "Repeat ... until |v^H H O|^2 is maximised"
        old = best
        m = [np.abs(probe(ch, x, q, rg, ctr)) for x in V]; i = int(np.argmax(m)); v, best = V[i], max(best, m[i])
        m = [np.abs(probe(ch, v, x, rg, ctr)) for x in W]; i = int(np.argmax(m)); q, best = W[i], max(best, m[i])
        ctr['iters'] += 1
        if best <= old * (1 + 1e-6): break
    return v, q

# ------------------------------------------------------------------ proposed UL-JBA (Algorithm 1, modified)
def shortlist(grid, sig_ang, u0, pos, S, rg, n_samp=24):
    # rank candidate beams by their *expected* normalised array gain under the estimator's direction uncertainty
    e1, e2 = tangent(u0)
    smp = unit(u0 + sig_ang * (rg.normal(size=(n_samp, 1)) * e1 + rg.normal(size=(n_samp, 1)) * e2))
    A, B = steer(grid, pos), steer(smp, pos)
    sc = np.mean(np.abs(np.conj(A) @ B.T) ** 2, axis=1)
    return grid[np.argsort(-sc)[:S]]

def ul_jba(ch, p_est, sigma, rg, ctr, u_est=None, S_v=4, S_w=4, eps=0.05, max_it=4):
    s = ch['s']; d = np.linalg.norm(p_est); u0 = unit(p_est)
    sig_ang = max(float(np.mean(sigma)) / d, 0.01)                              # direction uncertainty [rad]
    rad = 2.0 * sig_ang
    gV, gW = dir_grid(u0, rad, STEP_RX), dir_grid(u0, rad, STEP_RIS)
    shV = shortlist(gV, sig_ang, u0, UPOS, S_v, rg); shW = shortlist(gW, sig_ang, u0, RPOS, S_w, rg)
    ctr['cls_flops'] += 8 * 24 * (len(gV) * Nr + len(gW) * Ms)
    q0 = quant(ris_phase_from_u(u_est, s)) if u_est is not None else quant(ris_phase(u0, s))   # closed-form start (Eq. 39/40)
    candV = [(rx_beam(u0), u0)] + [(rx_beam(x), x) for x in shV]
    candW = [(q0, u0)] + [(quant(ris_phase(x, s)), x) for x in shW]
    seen = set(); cw = []
    for q, x in candW:
        key = q.tobytes()
        if key not in seen: seen.add(key); cw.append((q, x))
    # (1) evaluate shortlisted Rx beams with the initial RIS configuration, (2) shortlisted RIS configs with the best Rx beam
    mv = [np.abs(probe(ch, v, q0, rg, ctr)) for v, _ in candV]; i = int(np.argmax(mv)); (v, uv), best = candV[i], mv[i]
    q, uq = q0, u0
    for qq, x in cw[1:]:
        m = np.abs(probe(ch, v, qq, rg, ctr))
        if m > best: q, uq, best = qq, x, m
    # (3) local alternatives around the incumbent; stop when the achievable-rate gain of a round falls below eps
    r_old = rate_probe(best)
    for _ in range(max_it):
        e1, e2 = tangent(uv); 
        for dv in (STEP_RX * e1, -STEP_RX * e1, STEP_RX * e2, -STEP_RX * e2):
            x = unit(uv + dv); vv = rx_beam(x); m = np.abs(probe(ch, vv, q, rg, ctr))
            if m > best: v, uv, best = vv, x, m
        e1, e2 = tangent(uq)
        for dq in (STEP_RIS * e1, -STEP_RIS * e1, STEP_RIS * e2, -STEP_RIS * e2):
            x = unit(uq + dq); qq = quant(ris_phase(x, s)); m = np.abs(probe(ch, v, qq, rg, ctr))
            if m > best: q, uq, best = qq, x, m
        ctr['iters'] += 1; r_new = rate_probe(best)
        if r_new - r_old < eps: break
        r_old = r_new
    return v, q

## 6. Change 3 - Quantum-Assisted Discrete RIS Phase Optimisation (Q-RPO)
**Problem encoding.** The closed-form (quantised) configuration `q0` is retained as the start. The sub-RIS is split into `2 x 3` tiles; each tile may rotate its phases by `k*pi/2` (k = 0..3), i.e. **2 qubits per tile -> 12 qubits, 4096 candidate configurations**. The effective gain is linear in the tile phasors, so

`f(x) = | sum_t exp(j*pi*c_t(x)/2) * A_t |^2`, a pseudo-Boolean (degree-4) function of the 12 bits.

`A_t` is **measured**, not assumed: probe `q0` and `q0` with tile `t` flipped by pi (`A_t = (g0 - g_t)/2`, denoised with `sum_t A_t = g0`) - `tiles + 1 = 7` probes.

**Quantum search.**
* **QAOA** (p = 2): cost unitary `exp(-i gamma (1 - f_norm))`, mixer `RX(2 beta)`, COBYLA over (gamma, beta), then `shots` samples; the best sampled bit-string (never worse than `q0`, verified by one probe) is applied.
* **VQC**: hardware-efficient ansatz (RY-RZ + CZ ring, 2 layers) trained to minimise `<1 - f_norm>`.
* The Walsh-Hadamard transform of `f` gives the **Pauli-Z terms** (used for the gate counts in Section 9).

**Classical comparison on the same candidate space:** exhaustive search (global optimum of the surrogate), tile-wise greedy coordinate descent, simulated annealing and random search with the same evaluation budget.

In [ ]:
# ======================================================================= Q-RPO: quantum-assisted discrete RIS phase optimisation
IDX = np.arange(2 ** NQ)
BITS = ((IDX[:, None] >> (NQ - 1 - np.arange(NQ))) & 1)            # qubit 0 = most significant bit
CHOICE = BITS[:, 0::2] * 2 + BITS[:, 1::2]                          # 2 qubits -> one of 4 phase offsets (k*pi/2) per tile
PHASE_TILE = np.exp(1j * np.pi / 2 * CHOICE)                        # (2^NQ, NT)

def apply_tiles(q0, x):                                             # candidate configuration for bit-string index x
    return q0 * PHASE_TILE[x][TILE]

def tile_phasors(ch, v, q0, rg, ctr, noisy=True):
    # tile-wise (group) training: flip tile t by pi and measure -> A_t = (g0 - g_t)/2 ; sum_t A_t = g0 is used to denoise
    if noisy:
        g0 = probe(ch, v, q0, rg, ctr); gt = []
        for t in range(NT):
            q = q0.copy(); q[TILE == t] *= -1; gt.append(probe(ch, v, q, rg, ctr))
        A = (g0 - np.array(gt)) / 2; A = A + (g0 - A.sum()) / NT
    else:
        A = np.array([gain(ch, v, q0 * (TILE == t)) for t in range(NT)])
    return A

def cost_vec(A): return np.abs(PHASE_TILE @ A) ** 2                 # f(x) = |sum_t e^{j pi c_t(x)/2} A_t|^2  (pseudo-Boolean in the 2*NT bits)

def pauli_terms(f):
    # Walsh-Hadamard transform: f(x) = sum_S c_S prod_{k in S} z_k ; returns #terms by Pauli-Z weight (gate-count input)
    c = f.astype(float).copy(); h = 1
    while h < len(c):
        c = c.reshape(-1, 2, h); c = np.stack([c[:, 0] + c[:, 1], c[:, 0] - c[:, 1]], 1).reshape(-1); h *= 2
    c /= len(c); w = np.array([bin(i).count('1') for i in range(len(c))]); keep = np.abs(c) > 1e-6 * np.abs(c).max()
    return {int(k): int(np.sum(keep & (w == k))) for k in range(1, 5)}

# ---- statevector simulator (exact; cost of a real device is accounted separately in the complexity section)
def rx_all(psi, theta, n):
    c, s = np.cos(theta / 2), -1j * np.sin(theta / 2)
    for k in range(n):
        v = psi.reshape(2 ** k, 2, -1); out = np.empty_like(v)
        out[:, 0] = c * v[:, 0] + s * v[:, 1]; out[:, 1] = s * v[:, 0] + c * v[:, 1]; psi = out.reshape(-1)
    return psi

def qaoa_state(x, cn, p):
    psi = np.full(2 ** NQ, 2 ** (-NQ / 2), complex)
    for l in range(p):
        psi = psi * np.exp(-1j * x[l] * cn); psi = rx_all(psi, 2 * x[p + l], NQ)
    return psi

def run_qaoa(f, p=2, iters=60):
    cn = 1 - (f - f.min()) / (f.max() - f.min() + 1e-30)            # H_C = 1 - f_norm  (ground state = best candidate)
    x0 = np.concatenate([(np.arange(p) + 0.5) / p * 0.9, (1 - (np.arange(p) + 0.5) / p) * 0.9])
    obj = lambda x: float(np.real(np.vdot(qaoa_state(x, cn, p), cn * qaoa_state(x, cn, p))))
    r = minimize(obj, x0, method='COBYLA', options=dict(maxiter=iters, rhobeg=0.3))
    return qaoa_state(r.x, cn, p), r.nfev, dict(p=p, depth_layers=p)

CZ_RING = np.ones(2 ** NQ)
for k in range(NQ): CZ_RING *= (-1.0) ** (BITS[:, k] * BITS[:, (k + 1) % NQ])

def one_q(psi, U, k):
    v = psi.reshape(2 ** k, 2, -1); out = np.empty_like(v)
    out[:, 0] = U[0, 0] * v[:, 0] + U[0, 1] * v[:, 1]; out[:, 1] = U[1, 0] * v[:, 0] + U[1, 1] * v[:, 1]; return out.reshape(-1)

def vqc_state(th, L):
    psi = np.full(2 ** NQ, 2 ** (-NQ / 2), complex); th = th.reshape(L, NQ, 2)
    for l in range(L):
        for k in range(NQ):
            a, b = th[l, k]; ry = np.array([[np.cos(a / 2), -np.sin(a / 2)], [np.sin(a / 2), np.cos(a / 2)]])
            rz = np.diag([np.exp(-1j * b / 2), np.exp(1j * b / 2)]); psi = one_q(psi, rz @ ry, k)
        psi = psi * CZ_RING
    return psi

def run_vqc(f, L=2, iters=200, rg=None):
    cn = 1 - (f - f.min()) / (f.max() - f.min() + 1e-30)
    x0 = (rg or np.random.default_rng(0)).normal(0, 0.1, L * NQ * 2)
    obj = lambda x: float(np.real(np.vdot(vqc_state(x, L), cn * vqc_state(x, L))))
    r = minimize(obj, x0, method='COBYLA', options=dict(maxiter=iters, rhobeg=0.5))
    return vqc_state(r.x, L), r.nfev, dict(L=L, depth_layers=L)

# ---- classical optimisers over the *same* candidate space (cost evaluations counted)
def cls_exhaustive(f): return int(np.argmax(f)), len(f)
def cls_greedy(f, sweeps=3):
    ch = np.zeros(NT, int); ev = 0
    def idx(c): return int(sum(int(c[t]) << (2 * (NT - 1 - t)) for t in range(NT)))
    for _ in range(sweeps):
        for t in range(NT):
            vals = []
            for a in range(4): c2 = ch.copy(); c2[t] = a; vals.append(f[idx(c2)]); ev += 1
            ch[t] = int(np.argmax(vals))
    return idx(ch), ev
def cls_sa(f, budget, rg):
    ch = np.zeros(NT, int); idx = lambda c: int(sum(int(c[t]) << (2 * (NT - 1 - t)) for t in range(NT)))
    cur = f[idx(ch)]; best, bi = cur, idx(ch); T0 = 0.15 * (f.max() - f.min() + 1e-30)
    for i in range(budget):
        c2 = ch.copy(); c2[rg.integers(NT)] = rg.integers(4); val = f[idx(c2)]; Tt = T0 * (1 - i / budget) + 1e-12
        if val >= cur or rg.random() < np.exp((val - cur) / Tt): ch, cur = c2, val
        if cur > best: best, bi = cur, idx(ch)
    return bi, budget
def cls_random(f, budget, rg):
    idx = rg.integers(0, len(f), budget); return int(idx[np.argmax(f[idx])]), budget

def q_rpo(ch, v, q0, rg, ctr, solver='qaoa', shots=256, p=2, iters=60, L=2, noisy=True):
    A = tile_phasors(ch, v, q0, rg, ctr, noisy=noisy); f = cost_vec(A); info = dict(solver=solver, f_opt=f.max(), f0=f[0])
    if solver in ('qaoa', 'vqc'):
        psi, nfev, meta = run_qaoa(f, p, iters) if solver == 'qaoa' else run_vqc(f, L, iters * 3, rg)
        pr = np.abs(psi) ** 2; pr /= pr.sum(); smp = np.unique(rg.choice(len(f), size=shots, p=pr))
        x = int(smp[np.argmax(f[smp])]); x = x if f[x] > f[0] else 0
        info.update(nfev=nfev, shots_total=(nfev + 1) * shots, meta=meta, cls_evals=len(smp), p_opt=float(pr[int(np.argmax(f))]))
    elif solver == 'exhaustive': x, ev = cls_exhaustive(f); info.update(cls_evals=ev)
    elif solver == 'greedy': x, ev = cls_greedy(f); info.update(cls_evals=ev)
    elif solver == 'sa': x, ev = cls_sa(f, shots, rg); info.update(cls_evals=ev)
    elif solver == 'random': x, ev = cls_random(f, shots, rg); info.update(cls_evals=ev)
    q = apply_tiles(q0, x)
    # one verification probe: keep the new configuration only if it measures better than the closed-form start
    if x != 0:
        gm = probe(ch, v, q, rg, ctr); g0 = probe(ch, v, q0, rg, ctr)
        if abs(gm) < abs(g0): q, x = q0, 0
    info['x'] = x; info['f_sel'] = f[x]; return q, info

## 7. End-to-end pipelines on the test users
Methods evaluated (each user, effective gain `|v^H H O G w|` measured on the **true** channel):

* *Random phase*, *UWB closed-form* (Eq. 40, continuous and 2-bit) - paper baselines without search
* **PBA (original)** - Algorithm 1
* **Lightweight (NN + UL-JBA)** - Changes 1 + 2
* **Quantum-assisted (UWB + Q-RPO)** - Change 3 applied to the original estimator (closed-form start, Rx beam from Eq. 34)
* **Full (NN + UL-JBA + Q-RPO)** - Changes 1 + 2 + 3
* *Genie* (true channel, continuous and 2-bit) - upper bounds

In [ ]:
Tn = cfg.T + cfg.horizon - 1
NU, KK = len(te['P']), cfg.K
pred = predict_full(te)                         # proposed estimator output for every test user
METHODS = ['Random phase', 'UWB closed-form (cont.)', 'UWB closed-form (2-bit)', 'PBA (original)',
           'Lightweight (NN + UL-JBA)', 'Quantum-assisted (UWB + Q-RPO)', 'Full (NN + UL-JBA + Q-RPO)',
           'Genie (2-bit)', 'Genie (cont.)']
R = {m: dict(V=np.zeros((NU, Nr), complex), Q=np.zeros((NU, Ms), complex), probes=np.zeros(NU), cls=np.zeros(NU),
             iters=np.zeros(NU), shots=np.zeros(NU), nfev=np.zeros(NU), qinfo=[None] * NU) for m in METHODS}

def store(m, i, v, q, ctr, info=None):
    R[m]['V'][i], R[m]['Q'][i] = v, q
    R[m]['probes'][i], R[m]['cls'][i], R[m]['iters'][i] = ctr['probes'], ctr['cls_flops'], ctr['iters']
    if info is not None:
        R[m]['qinfo'][i] = info; R[m]['shots'][i] = info.get('shots_total', 0); R[m]['nfev'][i] = info.get('nfev', 0)

rg_run = np.random.default_rng(SEED + 100)
t0 = time.time(); tm = {m: 0.0 for m in METHODS}
for i in range(NU):
    chu = pick(te['ch'], i); s = chu['s']; phat = te['phat'][i, -1]; u0 = unit(phat)
    u_true = unit(te['P'][i, Tn])
    ts = time.time()
    store('Random phase', i, rx_beam(u0), quant(np.exp(1j * rg_run.uniform(0, 2 * np.pi, Ms))), new_ctr())
    store('UWB closed-form (cont.)', i, rx_beam(u0), ris_phase(u0, s), new_ctr())
    q_uwb = quant(ris_phase(u0, s)); store('UWB closed-form (2-bit)', i, rx_beam(u0), q_uwb, new_ctr())
    c = new_ctr(); v, q = pba(chu, phat, rg_run, c); store('PBA (original)', i, v, q, c)
    # --- Change 3 only: original estimator + Q-RPO starting from the closed-form (Eq. 40) configuration
    c = new_ctr(); q2, info = q_rpo(chu, rx_beam(u0), q_uwb, rg_run, c, 'qaoa'); store('Quantum-assisted (UWB + Q-RPO)', i, rx_beam(u0), q2, c, info)
    # --- Changes 1+2: DS-CNN/GRU estimator + UL-JBA
    c = new_ctr(); v, q = ul_jba(chu, pred['p_est'][i], pred['sigma'][i], rg_run, c, pred['u_hat'][i]); store('Lightweight (NN + UL-JBA)', i, v, q, c)
    # --- Changes 1+2+3: add Q-RPO on top of the UL-JBA result (closed-form-type start, v fixed)
    c2 = dict(c); q3, info = q_rpo(chu, v, q, rg_run, c2, 'qaoa'); store('Full (NN + UL-JBA + Q-RPO)', i, v, q3, c2, info)
    # --- genie bounds
    store('Genie (cont.)', i, rx_beam(u_true), ris_phase_from_u(te['u'][i], s), new_ctr())
    store('Genie (2-bit)', i, rx_beam(u_true), quant(ris_phase_from_u(te['u'][i], s)), new_ctr())
    if i % 100 == 99: print(f"  processed {i+1}/{NU} users ({time.time()-t0:.0f}s)")

G = {m: np.array([np.abs(gain(pick(te['ch'], i), R[m]['V'][i], R[m]['Q'][i])) for i in range(NU)]) for m in METHODS}
g_ref = np.mean(G['Genie (cont.)'] ** 2)
print(f"\n{'method':34s} {'gain vs genie [dB]':>19s} {'probes/user':>12s}")
for m in METHODS: print(f"{m:34s} {10*np.log10(np.mean(G[m]**2)/g_ref):19.2f} {R[m]['probes'].mean():12.1f}")

## 8. System-level results: MMSE precoding and sum-rate (Eqs. 42-47)

In [ ]:
# ---- multi-user effective channel T (K x K), MMSE digital precoder (Eq. 42 form) and sum-rate (Eqs. 44-47)
def eff_matrix(m, g):
    Tm = np.zeros((KK, KK), complex)
    for k in range(KK):
        chk = pick(te['ch'], g * KK + k)
        for i in range(KK):                                  # user k hears the reflection of sub-RIS i (leakage terms of Eq. 46)
            Tm[k, i] = gain(chk, R[m]['V'][g * KK + k], R[m]['Q'][g * KK + i], s=S_BS[i])
    return Tm

def sum_rate(Tm, p_dbm):
    P = 10 ** (p_dbm / 10) * 1e-3
    F = Tm.conj().T @ np.linalg.inv(Tm @ Tm.conj().T + KK * SIG2 / P * np.eye(KK))
    F = F / np.linalg.norm(F, axis=0, keepdims=True)                      # ||W f_k|| = 1  (Eq. 4)
    He = Tm @ F; sig = np.abs(np.diag(He)) ** 2; intf = np.sum(np.abs(He) ** 2, 1) - sig
    return np.sum(np.log2(1 + (P / KK) * sig / ((P / KK) * intf + SIG2)))

PW = np.arange(10, 45, 5); NS = NU // KK
TM = {m: [eff_matrix(m, g) for g in range(NS)] for m in METHODS}
SR = {m: np.array([[sum_rate(TM[m][g], p) for g in range(NS)] for p in PW]).mean(1) for m in METHODS}

fig, ax = plt.subplots(1, 2, figsize=(13, 4.6))
sty = {'Random phase': ('k', ':', 'x'), 'UWB closed-form (cont.)': ('tab:gray', '--', 's'), 'UWB closed-form (2-bit)': ('tab:brown', '--', 'v'),
       'PBA (original)': ('tab:blue', '-', 'o'), 'Lightweight (NN + UL-JBA)': ('tab:green', '-', '^'), 'Quantum-assisted (UWB + Q-RPO)': ('tab:orange', '-', 'D'),
       'Full (NN + UL-JBA + Q-RPO)': ('tab:red', '-', '*'), 'Genie (2-bit)': ('tab:purple', '-.', ''), 'Genie (cont.)': ('k', '-.', '')}
for m in METHODS:
    c_, l_, mk = sty[m]; ax[0].plot(PW, SR[m], color=c_, ls=l_, marker=mk, label=m)
ax[0].set_xlabel('Transmit power P [dBm]'); ax[0].set_ylabel('Sum-rate [bps/Hz]'); ax[0].grid(alpha=.3); ax[0].legend(fontsize=7)
ax[0].set_title(f'Sum-rate, K={KK} users, {cfg.mt}x{cfg.nt} BS, {cfg.ms}x{cfg.ns} sub-RIS, {cfg.mr}x{cfg.nr} user, {cfg.bits}-bit RIS')
for m in ['PBA (original)', 'Lightweight (NN + UL-JBA)', 'Quantum-assisted (UWB + Q-RPO)', 'Full (NN + UL-JBA + Q-RPO)', 'Genie (2-bit)']:
    x = np.sort(10 * np.log10(G[m] ** 2 / g_ref)); ax[1].plot(x, np.arange(1, len(x) + 1) / len(x), color=sty[m][0], ls=sty[m][1], label=m)
ax[1].set_xlabel('Per-user effective gain relative to genie mean [dB]'); ax[1].set_ylabel('CDF'); ax[1].grid(alpha=.3); ax[1].legend(fontsize=7)
plt.tight_layout(); plt.show()
print(f"{'method':34s}" + ''.join(f"{p:>8d}" for p in PW) + '   (sum-rate, bps/Hz)')
for m in METHODS: print(f"{m:34s}" + ''.join(f"{v:8.2f}" for v in SR[m]))

### 8b. Q-RPO solver study: QAOA vs VQC vs classical optimisers on the same 4096-state space

In [ ]:
# ---- Q-RPO solver study on the SAME candidate space (4^6 = 4096 tile-phase configurations), starting from the closed-form start
NSOL = 40 if FAST else 120
rg_s = np.random.default_rng(SEED + 5)
SOLVERS = ['qaoa', 'vqc', 'exhaustive', 'greedy', 'sa', 'random']
S = {k: dict(ratio=[], gain=[], evals=[], shots=[], t=[], popt=[]) for k in SOLVERS}
base, orc = [], []
for i in range(NSOL):
    chu = pick(te['ch'], i); s = chu['s']; u0 = unit(te['phat'][i, -1]); v = rx_beam(u0); q0 = quant(ris_phase(u0, s))
    Atrue = tile_phasors(chu, v, q0, rg_s, new_ctr(), noisy=False); fo = cost_vec(Atrue)
    base.append(abs(gain(chu, v, q0))); orc.append(abs(gain(chu, v, apply_tiles(q0, int(np.argmax(fo))))))
    for sol in SOLVERS:
        c = new_ctr(); ts = time.time(); q2, info = q_rpo(chu, v, q0, rg_s, c, sol)
        S[sol]['t'].append(time.time() - ts); S[sol]['gain'].append(abs(gain(chu, v, q2))); S[sol]['ratio'].append(fo[info['x']] / fo.max())
        S[sol]['evals'].append(info.get('cls_evals', 0)); S[sol]['shots'].append(info.get('shots_total', 0)); S[sol]['popt'].append(info.get('p_opt', np.nan))
gb = np.mean(np.array(base) ** 2)
print(f"{'solver':12s} {'true-gain gain over start [dB]':>31s} {'f(x)/f(x*) (true phasors)':>26s} {'cost evals':>11s} {'shots':>8s} {'P(opt) in final state':>22s} {'sim time [s]':>13s}")
for sol in SOLVERS:
    a = S[sol]; print(f"{sol:12s} {10*np.log10(np.mean(np.array(a['gain'])**2)/gb):31.2f} {np.mean(a['ratio']):26.3f} {np.mean(a['evals']):11.0f} {np.mean(a['shots']):8.0f} {np.nanmean(a['popt']) if sol in ('qaoa','vqc') else float('nan'):22.4f} {np.mean(a['t']):13.2f}")
print(f"{'oracle':12s} {10*np.log10(np.mean(np.array(orc)**2)/gb):31.2f}   (exhaustive on noise-free tile phasors)")
print(f"random-guess probability of hitting the optimum: 1/{2**NQ} = {1/2**NQ:.5f}")

## 9. Change 4 - Computational complexity, quantum execution cost and lightweight-inference analysis
For **PBA**, **lightweight-only**, **quantum-assisted** and the **full framework** the table accounts for

* *classical pre-processing*: UWB multilateration, feature/map construction, NN inference (CNN + GRU + MC-dropout heads), shortlist scoring, probe post-processing, Q-RPO cost-vector construction + Walsh-Hadamard decomposition + optimiser book-keeping;
* *quantum circuit execution*: number of circuits (optimiser evaluations + 1), **measurement repetitions (shots)**, gate counts derived from the Pauli-Z weight histogram, QPU time;
* *over-the-air cost*: probes/beam-training slots and RIS reconfigurations;
* *communication overhead*: RIS configuration words + feedback per probe, UWB reports, tile-phasor upload / bit-string download for the QPU offload.

All hardware speeds are explicit model inputs in dictionary `A` (edit them to match your platform); probe counts, FLOP counts, shots and gate counts come from the executed algorithms.

In [ ]:
# ================= Change 4: computational, quantum-execution and lightweight-inference cost =================
# ---- hardware / link assumptions (edit freely; every number below is a model input, not a measurement of a real device)
A = dict(cpu_gflops=20.0,      # edge/BS processor, effective GFLOP/s
         t_probe_us=50.0,      # one beam-training probe: RIS reconfiguration + pilot averaging
         t_shot_us=100.0,      # QPU repetition period (reset + circuit + readout) per shot
         t_1q_ns=30.0, t_2q_ns=300.0,
         link_mbps=10.0,       # RIS-controller / feedback / QPU-offload control link
         t_queue_s=(0.0, 1.0)) # quantum job queue latency: dedicated QPU vs shared cloud QPU

# ---- circuit statistics of the Q-RPO cost Hamiltonian (Pauli-Z weight histogram -> gate counts)
wt = {1: 0, 2: 0, 3: 0, 4: 0}
for i in range(20):
    chu = pick(te['ch'], i); u0 = unit(te['phat'][i, -1]); v = rx_beam(u0); q0 = quant(ris_phase(u0, chu['s']))
    for k, n_ in pauli_terms(cost_vec(tile_phasors(chu, v, q0, rg_run, new_ctr(), noisy=False))).items(): wt[k] += n_ / 20
P_QAOA = 2
n_rz = sum(wt.values()); n_cx = sum(n * 2 * (w - 1) for w, n in wt.items())
g1q = P_QAOA * (n_rz + NQ) + NQ; g2q = P_QAOA * n_cx                     # + initial Hadamard layer
t_circ_us = (g1q * A['t_1q_ns'] + g2q * A['t_2q_ns']) / 1e3
print(f"Q-RPO circuit: {NQ} qubits, Pauli-Z terms/weight {dict((k, round(v,1)) for k, v in wt.items())}, QAOA p={P_QAOA}")
print(f"  gates/circuit: {g1q:.0f} one-qubit + {g2q:.0f} CNOT (all-to-all connectivity assumed) -> gate time {t_circ_us:.1f} us ; shot period {A['t_shot_us']:.0f} us")

# ---- per-method cost model
macs_prop = macs_estimator(True, True); macs_std = macs_estimator(False, True)
def classical_flops(m, i):
    f = R[m]['cls'][i]
    uses_nn = m.startswith('Lightweight') or m.startswith('Full')
    uses_uwb = m not in ('Random phase', 'Genie (2-bit)', 'Genie (cont.)')
    reports = cfg.T if uses_nn else 1
    if uses_uwb: f += reports * 3.0e3                                    # Gauss-Newton multilateration, Eq. (24)
    if uses_nn: f += cfg.T * (Ms * 30 + 200) + 2 * macs_prop + 8 * 64 * (2 * Ms + 6) * 2     # features/maps + CNN-GRU + MC-dropout heads
    if m in ('Quantum-assisted (UWB + Q-RPO)', 'Full (NN + UL-JBA + Q-RPO)'):
        info = R[m]['qinfo'][i]
        f += (2 ** NQ) * NT * 8 + NQ * 2 ** NQ * 2 + R[m]['nfev'][i] * 400 + (2 ** NQ) * 4 + info.get('cls_evals', 0) * 20   # cost vector, WHT, COBYLA book-keeping, sample ranking
    return f

def comm_bits(m, i):
    b = R[m]['probes'][i] * (Ms * cfg.bits + 16)                         # RIS configuration word + 16-bit measured gain per probe
    if m not in ('Random phase', 'Genie (2-bit)', 'Genie (cont.)'): b += (cfg.T if m.startswith(('Light', 'Full')) else 1) * 4 * 16   # UWB ranges
    if R[m]['qinfo'][i] is not None: b += NT * 64 + NQ                   # tile phasors up, selected bit-string back
    return b

ROWS = ['PBA (original)', 'Lightweight (NN + UL-JBA)', 'Quantum-assisted (UWB + Q-RPO)', 'Full (NN + UL-JBA + Q-RPO)']
rec = {}
for m in ROWS:
    fl = np.mean([classical_flops(m, i) for i in range(NU)]); pr = R[m]['probes'].mean(); cb = np.mean([comm_bits(m, i) for i in range(NU)])
    sh = R[m]['shots'].mean(); nf = R[m]['nfev'].mean()
    t_cpu = fl / (A['cpu_gflops'] * 1e9) * 1e3; t_pr = pr * A['t_probe_us'] / 1e3; t_cm = cb / (A['link_mbps'] * 1e6) * 1e3
    t_q = sh * max(A['t_shot_us'], t_circ_us) / 1e3
    rec[m] = dict(flops=fl, probes=pr, bits=cb, shots=sh, circuits=(nf + 1) if sh else 0, t_cpu=t_cpu, t_probe=t_pr, t_comm=t_cm, t_q=t_q,
                  total0=t_cpu + t_pr + t_cm + t_q, total1=t_cpu + t_pr + t_cm + t_q + (A['t_queue_s'][1] * 1e3 if sh else 0),
                  gain=10 * np.log10(np.mean(G[m] ** 2) / g_ref), sr=SR[m][-1])

hdr = f"{'method':32s} {'probes':>7s} {'classical':>10s} {'circuits':>9s} {'shots':>8s} {'comm':>8s} | {'CPU':>7s} {'probe':>7s} {'comm':>7s} {'QPU':>8s} | {'total':>8s} {'+1s queue':>10s} | {'gain':>6s} {'SR@40':>6s}"
print('\n' + hdr); print(f"{'':32s} {'':>7s} {'[kFLOP]':>10s} {'':>9s} {'':>8s} {'[bits]':>8s} | {'[ms]':>7s} {'[ms]':>7s} {'[ms]':>7s} {'[ms]':>8s} | {'[ms]':>8s} {'[ms]':>10s} | {'[dB]':>6s} {'bps/Hz':>6s}")
for m, r in rec.items():
    print(f"{m:32s} {r['probes']:7.1f} {r['flops']/1e3:10.1f} {r['circuits']:9.0f} {r['shots']:8.0f} {r['bits']:8.0f} | {r['t_cpu']:7.3f} {r['t_probe']:7.2f} {r['t_comm']:7.3f} {r['t_q']:8.1f} | {r['total0']:8.2f} {r['total1']:10.1f} | {r['gain']:6.2f} {r['sr']:6.2f}")
print(f"\nLightweight inference: proposed DS-CNN+GRU = {abl['Proposed (DS-CNN + GRU)']['params']:,} params, {macs_prop/1e6:.2f} MMAC/inference; standard-conv variant = {macs_std/1e6:.2f} MMAC ({macs_std/macs_prop:.1f}x more)")
print(f"Measurement repetitions in Q-RPO: (optimiser evaluations + 1) x {256} shots; on a QPU this is the dominant cost term -> see the QPU column.")
print("Break-even note: the quantum stage is only worth its QPU time if it saves more than it costs in probes/latency (compare the 'total' columns).")

# ---- figure 1: latency breakdown ; figure 2: how the search effort scales with the RIS size
fig, ax = plt.subplots(1, 2, figsize=(13, 4.6))
labs = [m.replace(' (', '\n(') for m in rec]; bot = np.zeros(len(rec))
for key, col, nm in [('t_cpu', 'tab:blue', 'classical compute'), ('t_probe', 'tab:green', 'probing / RIS reconfig.'), ('t_comm', 'tab:gray', 'communication'), ('t_q', 'tab:red', 'QPU shots (dedicated)')]:
    v_ = np.array([rec[m][key] for m in rec]); ax[0].bar(labs, v_, bottom=bot, color=col, label=nm); bot += v_
ax[0].set_ylabel('latency per user [ms]'); ax[0].set_title('Alignment latency breakdown (model inputs in dict A)'); ax[0].legend(fontsize=8); ax[0].tick_params(axis='x', labelsize=7)
Ns = np.array([4, 8, 12, 16, 24, 32, 48, 64]); d_ = 1.4
def pba_probes(N, iters=3):
    n = int(np.ceil(cfg.r_e / d_ / (STEP_RIS * cfg.ms / N))); nw = 1 + sum(1 for i in range(-n, n + 1) for j in range(-n, n + 1) if i * i + j * j <= n * n)
    nv = 13; return 1 + iters * (nv + nw)
ax[1].semilogy(Ns, Ns ** 2 + Ns ** 4, 'k--', label='exhaustive search  N^2+N^4 (Table II)')
ax[1].semilogy(Ns, 18 * Ns + 12 * np.log(Ns) / np.log(3) - 3, color='gray', ls=':', label='tree-dictionary [19]')
ax[1].semilogy(Ns, [pba_probes(n) for n in Ns], 'o-', color='tab:blue', label='PBA (grows with codebook density)')
ax[1].semilogy(Ns, np.full(len(Ns), R['Lightweight (NN + UL-JBA)']['probes'].mean()), 's-', color='tab:green', label='UL-JBA (shortlist + local; N-independent)')
ax[1].semilogy(Ns, np.full(len(Ns), NT + 3), 'D-', color='tab:orange', label=f'Q-RPO probes (tiles+3; qubits fixed at {NQ})')
ax2 = ax[1].twinx(); ax2.plot(Ns, [2 * macs_estimator(True, True) * (n / cfg.ms) ** 2 / 1e6 for n in Ns], 'r-.', label='NN inference [MFLOP]'); ax2.set_ylabel('NN MFLOP / inference', color='r')
ax[1].set_xlabel('RIS elements per side N'); ax[1].set_ylabel('probes / beam-training slots'); ax[1].grid(alpha=.3, which='both'); ax[1].legend(fontsize=7, loc='upper left'); ax[1].set_title('Search effort vs RIS size')
plt.tight_layout(); plt.show()

## 10. Summary against the base paper

In [ ]:
# ---- final comparison against the base paper (all numbers from the runs above)
base_m, full_m, lite_m, quan_m = 'PBA (original)', 'Full (NN + UL-JBA + Q-RPO)', 'Lightweight (NN + UL-JBA)', 'Quantum-assisted (UWB + Q-RPO)'
print('=' * 100)
print(f"Probes per user          : PBA {R[base_m]['probes'].mean():.1f} | Lightweight {R[lite_m]['probes'].mean():.1f} ({100*(1-R[lite_m]['probes'].mean()/R[base_m]['probes'].mean()):.0f}% fewer) | Quantum-assisted {R[quan_m]['probes'].mean():.1f} | Full {R[full_m]['probes'].mean():.1f}")
print(f"Gain vs genie            : PBA {rec[base_m]['gain']:.2f} dB | Lightweight {rec[lite_m]['gain']:.2f} dB | Quantum-assisted {rec[quan_m]['gain']:.2f} dB | Full {rec[full_m]['gain']:.2f} dB")
print(f"Sum-rate @ 40 dBm        : PBA {SR[base_m][-1]:.2f} | Lightweight {SR[lite_m][-1]:.2f} | Quantum-assisted {SR[quan_m][-1]:.2f} | Full {SR[full_m][-1]:.2f}   (genie 2-bit {SR['Genie (2-bit)'][-1]:.2f}) bps/Hz")
print(f"Failures of the PBA error-sphere assumption (error > r_e): {np.mean(err_apply > cfg.r_e):.1%} of users; the UL-JBA search region adapts to the learned sigma instead.")
print('Caveat: the quantum stage is simulated exactly on a classical machine (no hardware noise); with a 12-qubit/4096-state space an exhaustive classical search is trivial,')
print('so Q-RPO here demonstrates a pipeline and cost accounting, not a quantum speed-up.')

## Reading the results honestly
* **What Changes 1-2 do:** the learned estimator denoises and *predicts* the UWB position/channel, and its own uncertainty lets UL-JBA search a region sized to the actual error instead of a fixed `r_e`. The gain is mainly **far fewer probes at equal or better rate**, and graceful behaviour when the UWB error exceeds the PBA sphere.
* **What Change 3 does:** starting from the closed-form 2-bit configuration, Q-RPO recovers a part of the quantisation/mis-pointing loss using only `tiles + 1` probes (see the *Quantum-assisted* row). On top of UL-JBA the incremental gain is small, because UL-JBA has already removed most of the pointing error: the table shows this instead of hiding it.
* **Quantum vs classical:** QAOA/VQC reach near-optimal candidates, and the probability of the exact optimum is far above random guessing, but with 12 bits an exhaustive classical search is trivial. Hardware shot cost (the QPU column) dominates the latency; Q-RPO only pays off for larger candidate spaces or if shots are much cheaper than assumed. The simulation is noise-free; real devices add gate/readout error.
* **Extending the study:** raise `FAST = False`, increase `tile_r/tile_c` (qubits grow, state-vector cost is 2^n), set `bits` to 1 or 3, change `uwb_sigma`, or swap `horizon` to test mobility robustness.